# Pan-human Azimuth label inventory

Confirm that the Zenodo v0 and v1 models carry every annotation level, then list every cell type the package can assign.

## Outline

1. **Models.** Download the v0 and v1 weights, check their md5 sums, and list each softmax head.
2. **Hierarchy.** Read the label encoders, the broad/medium/fine tables, and the Cell Ontology map, then confirm each head matches its encoder.
3. **Inventory.** Write one row per cell type, add Cell Ontology terms, and compare v0 with v1.
4. **Preprint.** Compare those counts with the bioRxiv paper.
5. **Table.** Show the full cell-type list.
6. **Immune cells.** Show only the rows whose broad label is Immune cell.

In [1]:
import sys
from pathlib import Path
import importlib

root = Path.cwd()
if not (root / "Python").is_dir():
    root = Path.cwd() / "pan_human_azimuth"
sys.path.insert(0, str(root / "Python"))

configure_paths = importlib.import_module("00_configure_paths")
paths = configure_paths.configure_paths(root)
paths

{'root': PosixPath('/data/nsiegel/projects/2026_11_18_pan_human_azimuth'),
 'python': PosixPath('/data/nsiegel/projects/2026_11_18_pan_human_azimuth/Python'),
 'models': PosixPath('/data/nsiegel/projects/2026_11_18_pan_human_azimuth/models'),
 'tables': PosixPath('/data/nsiegel/projects/2026_11_18_pan_human_azimuth/tables'),
 'paper': PosixPath('/data/nsiegel/projects/2026_11_18_pan_human_azimuth/paper')}

## 1. Models

### Download

Download the v0 and v1 weights from Zenodo record 20401417.

In [2]:
import importlib

download_models = importlib.import_module("01_download_models")
model_files = download_models.download_models(paths)
model_files

{'v0': PosixPath('/data/nsiegel/projects/2026_11_18_pan_human_azimuth/models/panhumanpy_inference_model_v0.keras'),
 'v1': PosixPath('/data/nsiegel/projects/2026_11_18_pan_human_azimuth/models/panhumanpy_inference_model_v1.keras')}

### Checksums

Check the downloaded files against the Zenodo md5 sums.

In [3]:
import importlib

verify_checksums = importlib.import_module("02_verify_checksums")
checksums = verify_checksums.verify_checksums(paths)
checksums

,model_version,path,expected_md5,observed_md5,match
0,v0,/data/nsiegel/projects/2026_11_18_pan_human_az...,767643b3e6cd65c24f64fbd355c13f27,767643b3e6cd65c24f64fbd355c13f27,True
1,v1,/data/nsiegel/projects/2026_11_18_pan_human_az...,e640b6031aa9a7ff7499df724f4abee0,e640b6031aa9a7ff7499df724f4abee0,True


### Softmax heads

List each softmax head in the Keras models.

In [4]:
import importlib

list_model_outputs = importlib.import_module("03_list_model_outputs")
model_outputs = list_model_outputs.list_model_outputs(paths)
model_outputs

,model_version,output_name,level_number,units
0,v0,L1_out,1,12
1,v0,L2_out,2,90
2,v0,L3_out,3,207
3,v0,L4_out,4,311
4,v0,L5_out,5,353
5,v0,L6_out,6,372
6,v0,L7_out,7,378
7,v0,L8_out,8,380
8,v1,L1_out,1,13
9,v1,L2_out,2,91


## 2. Hierarchy

### Label encoders

Read the package label encoders, the broad/medium/fine tables, and the Cell Ontology map.

In [5]:
import importlib

load_label_hierarchy = importlib.import_module("04_load_label_hierarchy")
hierarchies = load_label_hierarchy.load_label_hierarchy()
{
    version: [len(level["labels"]) for level in hierarchy["levels"]]
    for version, hierarchy in hierarchies.items()
}

{'v0': [12, 90, 207, 311, 353, 372, 378, 380],
 'v1': [13, 91, 209, 313, 355, 374, 380, 382]}

### Confirm levels

Confirm that every head size matches its encoder, and that broad, medium, fine, and Cell Ontology columns are present.

In [6]:
import importlib

confirm_annotation_levels = importlib.import_module("05_confirm_annotation_levels")
level_checks = confirm_annotation_levels.confirm_annotation_levels(
    model_outputs, hierarchies
)
level_checks

,model_version,check,expected,observed,passed
0,v0,eight_hierarchy_levels,8,8,True
1,v0,softmax_head_count,8,8,True
2,v0,L1_out_matches_encoder,12,12,True
3,v0,L2_out_matches_encoder,90,90,True
4,v0,L3_out_matches_encoder,207,207,True
5,v0,L4_out_matches_encoder,311,311,True
6,v0,L5_out_matches_encoder,353,353,True
7,v0,L6_out_matches_encoder,372,372,True
8,v0,L7_out_matches_encoder,378,378,True
9,v0,L8_out_matches_encoder,380,380,True


## 3. Inventory

### Cell types

Write one row per distinct hierarchical cell type.

In [7]:
import importlib

build_label_inventory = importlib.import_module("06_build_label_inventory")
inventory = build_label_inventory.build_label_inventory(hierarchies, paths)
inventory.groupby("model_version").size()

model_version
v0    445
v1    447
dtype: int64

### Cell Ontology

Map each final label to its Cell Ontology term.

In [8]:
import importlib

map_cell_ontology = importlib.import_module("07_map_cell_ontology")
inventory = map_cell_ontology.map_cell_ontology(inventory, hierarchies, paths)
inventory["cl_id"].eq("unmapped").groupby(inventory["model_version"]).sum()

model_version
v0    0
v1    1
Name: cl_id, dtype: int64

### v0 versus v1

Count labels at each level and list labels added or removed between v0 and v1.

In [9]:
import importlib

summarize_versions = importlib.import_module("08_summarize_versions")
summary = summarize_versions.summarize_versions(inventory, hierarchies, paths)
summary[
    summary["record_type"].isin(
        ["unique_labels", "head_classes", "added_in_v1", "removed_in_v1"]
    )
]

,record_type,model_version,label,count
0,unique_labels,v0,all_levels,445
1,head_classes,v0,level_1,12
2,head_classes,v0,level_2,90
3,head_classes,v0,level_3,207
4,head_classes,v0,level_4,311
5,head_classes,v0,level_5,353
6,head_classes,v0,level_6,372
7,head_classes,v0,level_7,378
8,head_classes,v0,level_8,380
348,unique_labels,v1,all_levels,447


## 4. Preprint

Compare those counts with the bioRxiv preprint.

In [10]:
import importlib

crosscheck_preprint = importlib.import_module("09_crosscheck_preprint")
crosscheck = crosscheck_preprint.crosscheck_preprint(
    inventory, hierarchies, model_outputs, paths
)
crosscheck

,claim,preprint_value,v0_value,v1_value,v0_match,v1_match,note
0,hierarchy_levels,8,8,8,True,True,Preprint: annotations at eight levels of resol...
1,level_1_classes,13,12,13,False,True,Preprint architecture table versus encoder siz...
2,level_2_classes,91,90,91,False,True,Preprint architecture table versus encoder siz...
3,level_3_classes,209,207,209,False,True,Preprint architecture table versus encoder siz...
4,level_4_classes,313,311,313,False,True,Preprint architecture table versus encoder siz...
5,level_5_classes,355,353,355,False,True,Preprint architecture table versus encoder siz...
6,level_6_classes,374,372,374,False,True,Preprint architecture table versus encoder siz...
7,level_7_classes,380,378,380,False,True,Preprint architecture table versus encoder siz...
8,level_8_classes,382,380,382,False,True,Preprint architecture table versus encoder siz...
9,training_tissues,23,,body_proper,False,False,23 tissues describes the training corpus. The ...


## 5. Full table

Full cell-type table, sorted by model version and broad label. The same table is saved at `tables/azimuth_label_inventory.tsv`.

In [11]:
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
inventory = pd.read_csv(paths["tables"] / "azimuth_label_inventory.tsv", sep="\t")
inventory.sort_values(["model_version", "azimuth_broad", "full_hierarchical_label"])

,model_version,hierarchy_level,full_hierarchical_label,level_0,level_1,level_2,level_3,level_4,level_5,level_6,level_7,final_level_label,azimuth_broad,azimuth_medium,azimuth_fine,cl_id,cl_name
0,v0,1,Adipocyte,Adipocyte,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Adipocyte,Adipocyte,Adipocyte,Adipocyte,CL:0000136,adipocyte
1,v0,1,Doublet like cell,Doublet like cell,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Doublet like cell,Doublet like cell,Doublet like cell,NaN,CL:0000000,cell:doublet like cell
2,v0,2,Doublet like cell|Fibroblast/Granulosa doublet like cell,Doublet like cell,Fibroblast/Granulosa doublet like cell,NaN,NaN,NaN,NaN,NaN,NaN,Fibroblast/Granulosa doublet like cell,Doublet like cell,Doublet like cell,Doublet like cell|Fibroblast/Granulosa doublet like cell,CL:0000000,cell:fibroblast/granulosa doublet like cell
3,v0,1,Embryonic cell,Embryonic cell,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Embryonic cell,Embryonic cell,Embryonic cell,NaN,CL:0002321,embryonic cell (metazoa)
4,v0,2,Embryonic cell|Mesodermal cell,Embryonic cell,Mesodermal cell,NaN,NaN,NaN,NaN,NaN,NaN,Mesodermal cell,Embryonic cell,Embryonic cell,NaN,CL:0000222,mesodermal cell
5,v0,3,Embryonic cell|Mesodermal cell|ZEB2+ mesoderm,Embryonic cell,Mesodermal cell,ZEB2+ mesoderm,NaN,NaN,NaN,NaN,NaN,ZEB2+ mesoderm,Embryonic cell,Embryonic cell,Embryonic cell|Mesodermal cell|ZEB2+ mesoderm,CL:0000222,mesodermal cell:ZEB2+
6,v0,2,Embryonic cell|Trophoblast,Embryonic cell,Trophoblast,NaN,NaN,NaN,NaN,NaN,NaN,Trophoblast,Embryonic cell,Embryonic cell,NaN,CL:0000351,trophoblast cell
7,v0,3,Embryonic cell|Trophoblast|Cytotrophoblast,Embryonic cell,Trophoblast,Cytotrophoblast,NaN,NaN,NaN,NaN,NaN,Cytotrophoblast,Embryonic cell,Embryonic cell,Embryonic cell|Trophoblast|Cytotrophoblast,CL:0000351,trophoblast cell:cytotrophoblast
8,v0,4,Embryonic cell|Trophoblast|Cytotrophoblast|PSG5+ syncytiotrophoblast,Embryonic cell,Trophoblast,Cytotrophoblast,PSG5+ syncytiotrophoblast,NaN,NaN,NaN,NaN,PSG5+ syncytiotrophoblast,Embryonic cell,Embryonic cell,Embryonic cell|Trophoblast|Cytotrophoblast,CL:0000525,syncytiotrophoblast cell:PSG5+
9,v0,4,Embryonic cell|Trophoblast|Cytotrophoblast|Villous cytotrophoblast,Embryonic cell,Trophoblast,Cytotrophoblast,Villous cytotrophoblast,NaN,NaN,NaN,NaN,Villous cytotrophoblast,Embryonic cell,Embryonic cell,Embryonic cell|Trophoblast|Cytotrophoblast,CL:2000060,placental villous trophoblast:cytotrophoblast


## 6. Immune cell table

Rows whose broad label is Immune cell, taken from `tables/azimuth_label_inventory.tsv`.

In [12]:
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
immune = pd.read_csv(paths["tables"] / "azimuth_label_inventory.tsv", sep="\t")
immune = immune.loc[immune["azimuth_broad"].eq("Immune cell")]
immune.sort_values(["model_version", "azimuth_medium", "full_hierarchical_label"])

,model_version,hierarchy_level,full_hierarchical_label,level_0,level_1,level_2,level_3,level_4,level_5,level_6,level_7,final_level_label,azimuth_broad,azimuth_medium,azimuth_fine,cl_id,cl_name
252,v0,2,Immune cell|Cycling immune cell,Immune cell,Cycling immune cell,NaN,NaN,NaN,NaN,NaN,NaN,Cycling immune cell,Immune cell,Immune cell|Cycling immune cell,Immune cell|Cycling immune cell,CL:0000738,leukocyte:cycling
254,v0,3,Immune cell|Erythrocyte/Megakaryocyte|Megakaryocyte,Immune cell,Erythrocyte/Megakaryocyte,Megakaryocyte,NaN,NaN,NaN,NaN,NaN,Megakaryocyte,Immune cell,Immune cell|Erythrocyte/Megakaryocyte|Megakaryocyte|Platelet,Immune cell|Erythrocyte/Megakaryocyte|Megakaryocyte|Platelet,CL:0000556,megakaryocyte
255,v0,4,Immune cell|Erythrocyte/Megakaryocyte|Megakaryocyte|Platelet,Immune cell,Erythrocyte/Megakaryocyte,Megakaryocyte,Platelet,NaN,NaN,NaN,NaN,Platelet,Immune cell,Immune cell|Erythrocyte/Megakaryocyte|Megakaryocyte|Platelet,Immune cell|Erythrocyte/Megakaryocyte|Megakaryocyte|Platelet,CL:0000233,platelet
262,v0,4,Immune cell|Erythrocyte/Megakaryocyte|Red blood cell|Fetal HBG+ erythrocyte,Immune cell,Erythrocyte/Megakaryocyte,Red blood cell,Fetal HBG+ erythrocyte,NaN,NaN,NaN,NaN,Fetal HBG+ erythrocyte,Immune cell,Immune cell|Erythrocyte/Megakaryocyte|Red blood cell,Immune cell|Erythrocyte/Megakaryocyte|Red blood cell|Fetal HBG+ erythrocyte,CL:0000232,erythrocyte:fetal
258,v0,5,Immune cell|Erythrocyte/Megakaryocyte|Red blood cell|Erythroblast|Early SOX4+ erythroblast,Immune cell,Erythrocyte/Megakaryocyte,Red blood cell,Erythroblast,Early SOX4+ erythroblast,NaN,NaN,NaN,Early SOX4+ erythroblast,Immune cell,Immune cell|Erythrocyte/Megakaryocyte|Red blood cell|Early SOX4+ erythroblast,Immune cell|Erythrocyte/Megakaryocyte|Red blood cell|Erythroblast|Early SOX4+ erythroblast,CL:0000765,erythroblast:early SOX4+
259,v0,5,Immune cell|Erythrocyte/Megakaryocyte|Red blood cell|Erythroblast|Intermediate EPCAM+ erythroblast,Immune cell,Erythrocyte/Megakaryocyte,Red blood cell,Erythroblast,Intermediate EPCAM+ erythroblast,NaN,NaN,NaN,Intermediate EPCAM+ erythroblast,Immune cell,Immune cell|Erythrocyte/Megakaryocyte|Red blood cell|Intermediate EPCAM+ erythroblast,Immune cell|Erythrocyte/Megakaryocyte|Red blood cell|Erythroblast|Intermediate EPCAM+ erythroblast,CL:0000765,erythroblast:intermediate EPCAM+
260,v0,5,Immune cell|Erythrocyte/Megakaryocyte|Red blood cell|Erythroblast|Late hemoglobin+ erythroblast,Immune cell,Erythrocyte/Megakaryocyte,Red blood cell,Erythroblast,Late hemoglobin+ erythroblast,NaN,NaN,NaN,Late hemoglobin+ erythroblast,Immune cell,Immune cell|Erythrocyte/Megakaryocyte|Red blood cell|Late hemoglobin+ erythroblast,Immune cell|Erythrocyte/Megakaryocyte|Red blood cell|Erythroblast|Late hemoglobin+ erythroblast,CL:0000765,erythroblast:late hemoglobin+
261,v0,6,Immune cell|Erythrocyte/Megakaryocyte|Red blood cell|Erythroblast|Late hemoglobin+ erythroblast|Cycling late hemoglo...,Immune cell,Erythrocyte/Megakaryocyte,Red blood cell,Erythroblast,Late hemoglobin+ erythroblast,Cycling late hemoglobin+ erythroblast,NaN,NaN,Cycling late hemoglobin+ erythroblast,Immune cell,Immune cell|Erythrocyte/Megakaryocyte|Red blood cell|Late hemoglobin+ erythroblast,Immune cell|Erythrocyte/Megakaryocyte|Red blood cell|Erythroblast|Late hemoglobin+ erythroblast,CL:0000765,erythroblast:cycling late hemoglobin+
265,v0,3,Immune cell|Hematopoietic precursor cell|EoBaMa Progenitor,Immune cell,Hematopoietic precursor cell,EoBaMa Progenitor,NaN,NaN,NaN,NaN,NaN,EoBaMa Progenitor,Immune cell,Immune cell|Hematopoietic precursor cell,Immune cell|Hematopoietic precursor cell|EoBaMa Progenitor,CL:0002191,granulocytopoietic cell:eosinophil-basophil-mast cell precursor
266,v0,3,Immune cell|Hematopoietic precursor cell|GMP,Immune cell,Hematopoietic precursor cell,GMP,NaN,NaN,NaN,NaN,NaN,GMP,Immune cell,Immune cell|Hematopoietic precursor cell,Immune cell|Hematopoietic precursor cell|GMP,CL:0000557,granulocyte monocyte progenitor cell
